In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import ast
from scipy import stats
import librosa
import warnings
import pandas as pd

In [ ]:
paths = {}
for root, dirs, files in os.walk(r"C:\Users\kinga\Desktop\studia\DS\ED_project\fma_small\fma_small"):
    if root.endswith("fma_small"):
        continue
    for file in files:
        full_path = os.path.join(root, file)
        track_id = int(file.split(".")[0])
        paths[track_id] = full_path

In [7]:
paths
len(list(paths.keys()))

8000

In [ ]:
def columns():
    feature_sizes = dict(
        chroma_stft=12,
        mfcc=20, rms=1, zcr=1,
        spectral_centroid=1
    )
    moments = ('mean', 'std')

    tuples = [
        (feature, moment, f"{i+1:02d}")
        for feature, size in feature_sizes.items()
        for moment in moments
        for i in range(size)
    ]

    return pd.MultiIndex.from_tuples(tuples, names=('feature', 'statistics', 'number')).sort_values()

In [ ]:
def compute_features(tid):

    features = pd.Series(index=columns(), dtype=np.float32, name=tid)

    # Catch warnings as exceptions (audioread leaks file descriptors).
    warnings.filterwarnings('error', module='librosa')

    def feature_stats(name, values):
        # values.shape = (num_rows, num_frames)
        for i in range(values.shape[0]):
            features[name, 'mean', f'{i+1:02d}'] = np.float32(np.mean(values[i, :]))
            features[name, 'std', f'{i+1:02d}'] = np.float32(np.std(values[i, :]))
            #features[name, 'skew', f'{i+1:02d}'] = np.float32(stats.skew(values[i, :]))
            #features[name, 'kurtosis', f'{i+1:02d}'] = np.float32(stats.kurtosis(values[i, :]))
            #features[name, 'median', f'{i+1:02d}'] = np.float32(np.median(values[i, :]))
            #features[name, 'min', f'{i+1:02d}'] = np.float32(np.min(values[i, :]))
            #features[name, 'max', f'{i+1:02d}'] = np.float32(np.max(values[i, :]))

    try:
        filepath = paths[tid]
        x, sr = librosa.load(filepath, sr=22050, mono=True)

        f = librosa.feature.zero_crossing_rate(x, frame_length=2048, hop_length=512)
        feature_stats('zcr', f)

        stft = np.abs(librosa.stft(x, n_fft=2048, hop_length=512))
        del x

        f = librosa.feature.chroma_stft(S=stft**2, n_chroma=12)
        feature_stats('chroma_stft', f)

        f = librosa.feature.rms(S=stft)
        feature_stats('rms', f)

        f = librosa.feature.spectral_centroid(S=stft)
        feature_stats('spectral_centroid', f)

        mel = librosa.feature.melspectrogram(sr=sr, S=stft**2)
        del stft
        f = librosa.feature.mfcc(S=librosa.power_to_db(mel), n_mfcc=20)
        feature_stats('mfcc', f)

    except Exception as e:
        print('{}: {}'.format(tid, repr(e)))
        
    return features

In [10]:
from multiprocessing.pool import ThreadPool
from tqdm import tqdm

track_ids = list(paths.keys())

features = pd.DataFrame(
    index=track_ids,
    columns=columns(),
    dtype=np.float32
)

nb_workers = int(1.5 * os.cpu_count())

pool = ThreadPool(nb_workers)

for row in tqdm(pool.imap_unordered(compute_features, track_ids), total=len(track_ids)):
    features.loc[row.name] = row

pool.close()
pool.join()


  0%|          | 0/8000 [00:00<?, ?it/s]C:\Users\Jakub Karczewski\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
 20%|█▉        | 1562/8000 [06:49<1:19:48,  1.34it/s]C:\Users\Jakub Karczewski\AppData\Local\Temp\ipykernel_26352\3981910055.py:13: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  features[name, 'skew', f'{i+1:02d}'] = np.float32(stats.skew(values[i, :]))
C:\Users\Jakub Karczewski\AppData\Local\Temp\ipykernel_26352\3981910055.py:14: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. R

41381: UserWarning('Trying to estimate tuning from empty frequency set.')


 25%|██▍       | 1965/8000 [10:09<24:19,  4.13it/s]  C:\Users\Jakub Karczewski\AppData\Local\Temp\ipykernel_26352\3981910055.py:13: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  features[name, 'skew', f'{i+1:02d}'] = np.float32(stats.skew(values[i, :]))
C:\Users\Jakub Karczewski\AppData\Local\Temp\ipykernel_26352\3981910055.py:14: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  features[name, 'kurtosis', f'{i+1:02d}'] = np.float32(stats.kurtosis(values[i, :]))
 38%|███▊      | 3023/8000 [14:11<08:40,  9.57it/s]  

69002: UserWarning('Trying to estimate tuning from empty frequency set.')


 42%|████▏     | 3334/8000 [15:23<10:58,  7.09it/s]C:\Users\Jakub Karczewski\AppData\Local\Temp\ipykernel_26352\3981910055.py:13: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  features[name, 'skew', f'{i+1:02d}'] = np.float32(stats.skew(values[i, :]))
C:\Users\Jakub Karczewski\AppData\Local\Temp\ipykernel_26352\3981910055.py:14: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  features[name, 'kurtosis', f'{i+1:02d}'] = np.float32(stats.kurtosis(values[i, :]))
 42%|████▏     | 3346/8000 [15:26<11:40,  6.64it/s]C:\Users\Jakub Karczewski\AppData\Local\Temp\ipykernel_26352\3981910055.py:13: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may

...

 71%|███████▏  | 5709/8000 [24:47<06:34,  5.80it/s]C:\Users\Jakub Karczewski\AppData\Local\Temp\ipykernel_26352\3981910055.py:13: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  features[name, 'skew', f'{i+1:02d}'] = np.float32(stats.skew(values[i, :]))
 71%|███████▏  | 5712/8000 [24:47<04:16,  8.91it/s]C:\Users\Jakub Karczewski\AppData\Local\Temp\ipykernel_26352\3981910055.py:14: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  features[name, 'kurtosis', f'{i+1:02d}'] = np.float32(stats.kurtosis(values[i, :]))
 76%|███████▌  | 6070/8000 [26:11<07:17,  4.42it/s]

123502: UserWarning('Trying to estimate tuning from empty frequency set.')


 77%|███████▋  | 6167/8000 [26:34<08:07,  3.76it/s]C:\Users\Jakub Karczewski\AppData\Local\Temp\ipykernel_26352\3981910055.py:13: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  features[name, 'skew', f'{i+1:02d}'] = np.float32(stats.skew(values[i, :]))
C:\Users\Jakub Karczewski\AppData\Local\Temp\ipykernel_26352\3981910055.py:14: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  features[name, 'kurtosis', f'{i+1:02d}'] = np.float32(stats.kurtosis(values[i, :]))
 87%|████████▋ | 6930/8000 [29:28<03:11,  5.58it/s]C:\Users\Jakub Karczewski\AppData\Local\Temp\ipykernel_26352\3981910055.py:21: UserWarning: PySoundFile failed. Trying audioread instead.
  x, sr = librosa.load(filepath, sr=22050, mono=True)  # kaiser_fast
 87%|████████▋ | 6931/8000 

133297: FutureWarning('librosa.core.audio.__audioread_load\n\tDeprecated as of librosa version 0.10.0.\n\tIt will be removed in librosa version 1.0.')


 92%|█████████▏| 7324/8000 [31:01<02:47,  4.04it/s]C:\Users\Jakub Karczewski\AppData\Local\Temp\ipykernel_26352\3981910055.py:13: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  features[name, 'skew', f'{i+1:02d}'] = np.float32(stats.skew(values[i, :]))
C:\Users\Jakub Karczewski\AppData\Local\Temp\ipykernel_26352\3981910055.py:14: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  features[name, 'kurtosis', f'{i+1:02d}'] = np.float32(stats.kurtosis(values[i, :]))
 93%|█████████▎| 7422/8000 [31:26<02:38,  3.65it/s]C:\Users\Jakub Karczewski\AppData\Local\Temp\ipykernel_26352\3981910055.py:13: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may

In [11]:
features.to_csv(
    "features_small3.csv",
    float_format="%.6f",
    header=['_'.join(col).strip() for col in features.columns.values],
    index_label="track_id"
)
features

feature    chroma_cens                                                     \
statistics    kurtosis                                                      
number              01        02        03        04        05         06   
2             0.007490 -0.520181  0.052345 -0.302098 -0.561455   0.456812   
5            -0.772786 -0.642317 -0.906739 -0.601648 -0.119200  -0.196957   
10            0.173267 -0.614480 -0.044852  0.444686  0.285313   0.322854   
140           0.051372  2.002213  0.265830 -0.017293 -0.430835  -1.261376   
141          -0.016673 -0.469405 -0.181815 -0.839419 -0.430258  -1.210200   
...                ...       ...       ...       ...       ...        ...   
154308       -1.326328  0.490313  1.925490 -0.866296  1.247579  -0.628696   
154309        1.990093 -1.199897 -0.073379 -1.005539 -0.976372  -0.508643   
154413       -1.254834 -1.375680  0.173443  4.630989 -0.613768   3.212755   
154414       -0.125803 -0.581913  1.337071 -0.700258 -1.310232  -1.111786   
155066        1.296299 -0.096545 -1.281948 -1.651980 -0.891290  12.386579   

feature                                             ...   tonnetz            \
statistics                                          ...       std             
number            07        08        09        10  ...        04        05   
2           1.779669  0.781414 -0.465594 -0.276902  ...  0.080555  0.021201   
5          -0.598001 -0.514312 -0.556515  1.947177  ...  0.085341  0.021255   
10         -0.655833 -0.500946  0.465562 -0.708733  ...  0.048777  0.021142   
140         0.801017 -0.037100 -0.356986 -1.217379  ...  0.206889  0.059962   
141        -0.419559 -0.749575  0.365205  0.625134  ...  0.175270  0.086150   
...              ...       ...       ...       ...  ...       ...       ...   
154308     -1.425239 -0.836263 -1.368011  2.880566  ...  0.158272  0.050959   
154309     -0.447739  7.300192 -0.047154  0.067279  ...  0.255038  0.046862   
154413      5.639235  9.353441 -1.510000  0.758934  ...  0.434680  0.093606   
154414     -1.143075  1.421414 -0.344749 -0.857938  ...  0.149456  0.062667   
155066      5.335883  0.522558 -0.840095  0.011947  ...  0.195379  0.203433   

feature                     zcr                                          \
statistics             kurtosis       max      mean    median       min   
number            06         01        01        01        01        01   
2           0.030955   4.087725  0.614746  0.164406  0.142578  0.030273   
5           0.020418   4.758883  0.487305  0.100544  0.087402  0.008301   
10          0.033729   2.607182  0.322266  0.148947  0.149414  0.070801   
140         0.080858  14.007420  0.398926  0.044525  0.026855  0.007812   
141         0.047561  33.820232  0.579590  0.062031  0.048340  0.012207   
...              ...        ...       ...       ...       ...       ...   
154308      0.107712   5.608341  0.398926  0.076531  0.074707  0.004883   
154309      0.069208   3.072596  0.584473  0.112923  0.084961  0.020020   
154413      0.128375   5.585150  0.229004  0.038495  0.030273  0.002930   
154414      0.065693   7.785815  0.474609  0.090261  0.072754  0.022461   
155066      0.153416  16.279776  0.090820  0.017363  0.016602  0.003906   

feature                         
statistics      skew       std  
number            01        01  
2           1.846496  0.093938  
5           1.711624  0.067248  
10          0.473020  0.028327  
140         3.475818  0.052389  
141         4.834568  0.052299  
...              ...       ...  
154308      1.874409  0.054476  
154309      1.750564  0.089936  
154413      1.725643  0.030414  
154414      2.312009  0.056155  
155066      2.297427  0.007670  

[8000 rows x 518 columns]